# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadaFouad461/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I build a 5-feature vector derived strictly from the February 2026 decision window: total impressions, total clicks, total sessions, average search position, and active reporting days. All features are aggregated at the client_hash_id × content_hash_id grain.

In [13]:
%pip -q install duckdb huggingface_hub pandas numpy

import getpass
import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import login, hf_hub_download, list_repo_files

# Authentication
HF_TOKEN = getpass.getpass("Enter Hugging Face Token: ")
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"

#  Download Data
all_files = list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
feb_files = [f for f in all_files if "month=2026-02" in f and f.endswith(".parquet")]
mar_files = [f for f in all_files if "month=2026-03" in f and f.endswith(".parquet")]

local_feb_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in feb_files]
local_mar_paths = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN) for f in mar_files]

#  Connection Setup
con = duckdb.connect()
FACT_FEB = f"read_parquet({local_feb_paths})"
FACT_MAR = f"read_parquet({local_mar_paths})"

print(" Connection & Data Ready!")

Enter Hugging Face Token: ··········
 Connection & Data Ready!


In [14]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(DISTINCT report_date) AS active_days_feb,
    SUM(gsc_clicks) AS total_clicks_feb,
    SUM(gsc_impressions) AS total_impressions_feb,
    AVG(gsc_avg_position) AS avg_position_feb,
    CASE
        WHEN SUM(gsc_impressions) > 0 THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr_feb
FROM {FACT_FEB}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

print("Feature frame shape:", feature_frame.shape)
display(feature_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (153559, 7)


,client_hash_id,content_hash_id,active_days_feb,total_clicks_feb,total_impressions_feb,avg_position_feb,ctr_feb
0,client_3ffa76342f366962,content_36bee0a093d0711d,2,0.0,3.0,5.500000,0.0
1,client_3ffa76342f366962,content_1546aabff77c05a4,4,1.0,5.0,5.000000,0.2
2,client_3ffa76342f366962,content_cae1d5374958a649,26,0.0,96.0,5.262333,0.0
3,client_3ffa76342f366962,content_dd66eecf9626cab8,28,0.0,235.0,6.407819,0.0
4,client_3ffa76342f366962,content_c51f1e8ef5502177,13,0.0,18.0,6.961538,0.0


### Feature notes

impressions_feb: Measures total observed search visibility in February 2026 prior to the prediction cutoff. Missing values default to 0.

clicks_feb: Measures total observed search clicks in February 2026 prior to the prediction cutoff. Missing values default to 0.

sessions_feb: Measures total organic/paid/social sessions in February 2026 prior to the prediction cutoff. Missing values default to 0.

avg_position_feb: Measures average SERP position during February 2026. Missing values are filled with the overall mean position.

active_days_feb: Measures the count of active data reporting days in February 2026. Missing values default to 0.

All five features originate strictly from February 2026 and precede the March outcome window.

In [15]:
# Missing values check and safe imputation
null_summary = feature_frame.isnull().sum().to_frame(name='missing_count')
display(null_summary)

# Safe imputation for average position if any nulls exist
if feature_frame['avg_position_feb'].isnull().sum() > 0:
    mean_pos = feature_frame['avg_position_feb'].mean()
    feature_frame['avg_position_feb'] = feature_frame['avg_position_feb'].fillna(mean_pos)

assert feature_frame.isnull().sum().sum() == 0
print("Feature hygiene check passed: Zero missing values.")

,missing_count
client_hash_id,0
content_hash_id,0
active_days_feb,0
total_clicks_feb,0
total_impressions_feb,0
avg_position_feb,0
ctr_feb,0


Feature hygiene check passed: Zero missing values.


## 3. The leakage hunt
I deliberately inject the future outcome (march_total_clicks from March 2026) as a feature into the February dataset. Because this metric comes from the post-decision window, its correlation with the target is exactly 1.0, proving intentional data leakage. I then drop this feature to keep the model honest.

In [16]:
# 1. Extract March Outcome (Future Target)
march_label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_total_clicks
FROM {FACT_MAR}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

# 2. Inject deliberate leakage
leaky_frame = feature_frame.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

leaky_frame["leaky_feature"] = leaky_frame["march_total_clicks"]

print("Rows in leakage evaluation:", len(leaky_frame))
print(
    "Correlation between leaky feature and March outcome:",
    leaky_frame["leaky_feature"].corr(leaky_frame["march_total_clicks"])
)

# 3. Clean up and remove leaky feature
clean_feature_frame = leaky_frame.drop(columns=["leaky_feature", "march_total_clicks"])
print("Leaky feature successfully removed. Clean frame shape:", clean_feature_frame.shape)

Rows in leakage evaluation: 134238
Correlation between leaky feature and March outcome: 1.0
Leaky feature successfully removed. Clean frame shape: (134238, 7)


## 4. What I excluded and why
client_hash_id: Excluded because it is a pseudonymous client identifier, not a predictive feature.

content_hash_id: Excluded because it identifies the content item without describing its underlying attributes.

march_total_clicks: Excluded from features because it belongs to the future outcome window (March 2026) and causes label leakage.

gsc_data_available IS FALSE rows: Excluded to eliminate non-reporting noisy data.

In [17]:
# Final confirmation of clean feature set
print("Final clean feature columns:", list(clean_feature_frame.columns))

Final clean feature columns: ['client_hash_id', 'content_hash_id', 'active_days_feb', 'total_clicks_feb', 'total_impressions_feb', 'avg_position_feb', 'ctr_feb']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.